In [3]:
pip list

Package                 Version
----------------------- -----------
asttokens               3.0.2
colorama                0.4.6
comm                    0.2.3
debugpy                 1.8.22
executing               2.2.1
ipykernel               7.4.0
ipython                 9.17.1
ipython_pygments_lexers 1.1.1
jedi                    0.20.0
jupyter_client          8.10.0
jupyter_core            5.9.1
matplotlib-inline       0.2.2
nest-asyncio2           1.7.3
packaging               26.3
parso                   0.8.7
pip                     26.2.1
platformdirs            4.12.2
prompt_toolkit          3.0.53
psutil                  7.2.2
pure_eval               0.2.4
Pygments                2.21.0
python-dateutil         2.9.0.post0
pyzmq                   27.2.0
six                     1.17.0
stack-data              0.6.3
tornado                 6.5.10
traitlets               5.16.1
typing_extensions       4.16.0
wcwidth                 0.9.1
Note: you may need to restart the kernel to 

In [4]:
pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 9.8/9.8 MB 52.9 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---------------------------------------- 12.7/12.7 MB 83.7 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

In [5]:
df_live = pd.read_csv("4G Movil_Provision_20260923153241.csv", low_memory=False)
df_guideline = pd.read_excel("A_PRACH CODIGOS FORMULA.xlsx", sheet_name='INFO DF')

In [6]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates radial distance in km between two lat/lon coordinates."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))


In [8]:
def plan_3sector_site(site_id, lat, lon, live_db=df_live, collision_km=8.0, confusion_km=12.0):
    # 1. Identify reserved PCIs and usable PCGroups from operator guidelines
    reserved_pcis = set(df_guideline[df_guideline['Group'].str.contains('Reservado')]['PCI'].unique())
    usable_df = df_guideline[~df_guideline['Group'].str.contains('Reservado')]
    usable_pcgroups = sorted(usable_df['PCGroup'].unique())
    
    # 2. Distance calculation across ALL active sectors
    live_db['dist'] = haversine_distance(live_db['LATITUD'], live_db['LONGITUD'], lat, lon)
    nearby_sectors = live_db[live_db['dist'] <= confusion_km]
    
    # 3. Build Blacklist: Reserved PCIs + nearby active PCIs
    blacklist = set(reserved_pcis)
    for pci in nearby_sectors['PHYCELLID'].dropna().astype(int):
        blacklist.add(pci)
        
    # Count Modulo 6 occurrences among close neighbors (<= collision_km)
    close_neighbors = nearby_sectors[nearby_sectors['dist'] <= collision_km]
    mod6_counts = {i: 0 for i in range(6)}
    for pci in close_neighbors['PHYCELLID'].dropna().astype(int):
        mod6_counts[pci % 6] += 1
        
    # 4. Find the optimal PCGroup
    best_candidate = None
    min_penalty = float('inf')
    
    for pcg in usable_pcgroups:
        group_rows = df_guideline[df_guideline['PCGroup'] == pcg].sort_values('PCI')
        pcis = group_rows['PCI'].tolist()
        rsis = group_rows['RSI'].tolist()
        grp_name = group_rows['Group'].iloc[0]
        
        p0, p1, p2 = pcis[0], pcis[1], pcis[2]
        
        # Ensure block is completely free of confusion conflicts
        if p0 in blacklist or p1 in blacklist or p2 in blacklist:
            continue
            
        # Score Modulo 6 RS collision penalty
        penalty = mod6_counts[p0 % 6] + mod6_counts[p1 % 6] + mod6_counts[p2 % 6]
        
        if penalty < min_penalty:
            min_penalty = penalty
            best_candidate = {
                'PCGroup': pcg,
                'Group_Name': grp_name,
                'PCIs': (p0, p1, p2),
                'RSIs': rsis,
                'Penalty': penalty
            }
            if penalty == 0:
                break
                
    if best_candidate:
        p0, p1, p2 = best_candidate['PCIs']
        r0, r1, r2 = best_candidate['RSIs']
        return {
            'SITE_ID': site_id,
            'LATITUDE': lat,
            'LONGITUDE': lon,
            'PCGroup': best_candidate['PCGroup'],
            'Group_Name': best_candidate['Group_Name'],
            'Sec1_PCI': p0, 'Sec1_RSI': r0, 'Sec1_Mod3': p0 % 3,
            'Sec2_PCI': p1, 'Sec2_RSI': r1, 'Sec2_Mod3': p1 % 3,
            'Sec3_PCI': p2, 'Sec3_RSI': r2, 'Sec3_Mod3': p2 % 3,
            'RS_Penalty': best_candidate['Penalty'],
            'STATUS': 'SUCCESS'
        }
    else:
        return {'SITE_ID': site_id, 'STATUS': 'FAILED_NO_CLEAN_GROUP_AVAILABLE'}

In [11]:
plan_3sector_site("tp0414", -5.19302, -80.5856 )

{'SITE_ID': 'tp0414',
 'LATITUDE': -5.19302,
 'LONGITUDE': -80.5856,
 'PCGroup': np.int64(35),
 'Group_Name': 'Group 2_33-62',
 'Sec1_PCI': 105,
 'Sec1_RSI': 120,
 'Sec1_Mod3': 0,
 'Sec2_PCI': 106,
 'Sec2_RSI': 128,
 'Sec2_Mod3': 1,
 'Sec3_PCI': 107,
 'Sec3_RSI': 136,
 'Sec3_Mod3': 2,
 'RS_Penalty': 421,
 'STATUS': 'SUCCESS'}

In [12]:
df_live2 = pd.read_csv("3G Movil_Provision_20260917104554.csv", low_memory=False)
df_guideline = pd.read_excel("A_PRACH CODIGOS FORMULA.xlsx", sheet_name='INFO DF')

In [13]:
def plan_3sector_3g(site_id, lat, lon, live_db=df_live2, collision_km=8.0, confusion_km=12.0):
    # 1. Identify reserved PCIs and usable PCGroups from operator guidelines
    reserved_pcis = set(df_guideline[df_guideline['Group'].str.contains('Reservado')]['PCI'].unique())
    usable_df = df_guideline[~df_guideline['Group'].str.contains('Reservado')]
    usable_pcgroups = sorted(usable_df['PCGroup'].unique())
    
    # 2. Distance calculation across ALL active sectors
    live_db['dist'] = haversine_distance(live_db['LATITUD'], live_db['LONGITUD'], lat, lon)
    nearby_sectors = live_db[live_db['dist'] <= confusion_km]
    
    # 3. Build Blacklist: Reserved PCIs + nearby active PCIs
    blacklist = set(reserved_pcis)
    for pci in nearby_sectors['PSCRAMBCODE'].dropna().astype(int):
        blacklist.add(pci)
        
    # Count Modulo 6 occurrences among close neighbors (<= collision_km)
    close_neighbors = nearby_sectors[nearby_sectors['dist'] <= collision_km]
    mod6_counts = {i: 0 for i in range(6)}
    for pci in close_neighbors['PSCRAMBCODE'].dropna().astype(int):
        mod6_counts[pci % 6] += 1
        
    # 4. Find the optimal PCGroup
    best_candidate = None
    min_penalty = float('inf')
    
    for pcg in usable_pcgroups:
        group_rows = df_guideline[df_guideline['PCGroup'] == pcg].sort_values('PCI')
        pcis = group_rows['PCI'].tolist()
        rsis = group_rows['RSI'].tolist()
        grp_name = group_rows['Group'].iloc[0]
        
        p0, p1, p2 = pcis[0], pcis[1], pcis[2]
        
        # Ensure block is completely free of confusion conflicts
        if p0 in blacklist or p1 in blacklist or p2 in blacklist:
            continue
            
        # Score Modulo 6 RS collision penalty
        penalty = mod6_counts[p0 % 6] + mod6_counts[p1 % 6] + mod6_counts[p2 % 6]
        
        if penalty < min_penalty:
            min_penalty = penalty
            best_candidate = {
                'PCGroup': pcg,
                'Group_Name': grp_name,
                'PCIs': (p0, p1, p2),
                'RSIs': rsis,
                'Penalty': penalty
            }
            if penalty == 0:
                break
                
    if best_candidate:
        p0, p1, p2 = best_candidate['PCIs']
        r0, r1, r2 = best_candidate['RSIs']
        return {
            'SITE_ID': site_id,
            'LATITUDE': lat,
            'LONGITUDE': lon,
            'PCGroup': best_candidate['PCGroup'],
            'Group_Name': best_candidate['Group_Name'],
            'Sec1_PCI': p0, 'Sec1_RSI': r0, 'Sec1_Mod3': p0 % 3,
            'Sec2_PCI': p1, 'Sec2_RSI': r1, 'Sec2_Mod3': p1 % 3,
            'Sec3_PCI': p2, 'Sec3_RSI': r2, 'Sec3_Mod3': p2 % 3,
            'RS_Penalty': best_candidate['Penalty'],
            'STATUS': 'SUCCESS'
        }
    else:
        return {'SITE_ID': site_id, 'STATUS': 'FAILED_NO_CLEAN_GROUP_AVAILABLE'}

In [14]:
plan_3sector_3g("upis", -5.19302, -80.5856)

{'SITE_ID': 'upis',
 'LATITUDE': -5.19302,
 'LONGITUDE': -80.5856,
 'PCGroup': np.int64(5),
 'Group_Name': 'Group 1_0-29',
 'Sec1_PCI': 15,
 'Sec1_RSI': 120,
 'Sec1_Mod3': 0,
 'Sec2_PCI': 16,
 'Sec2_RSI': 128,
 'Sec2_Mod3': 1,
 'Sec3_PCI': 17,
 'Sec3_RSI': 136,
 'Sec3_Mod3': 2,
 'RS_Penalty': 131,
 'STATUS': 'SUCCESS'}